# 12. Models and Tools — Deep Agents의 실행 표면 설계

## 학습 목표

- Deep Agents에서 **model**, **tool**, **permission boundary**가 맡는 역할을 구분합니다.
- 도구 docstring과 schema가 에이전트 행동에 주는 영향을 확인합니다.
- 실제 모델 호출 없이도 도구 명세와 위험도를 점검하는 방법을 익힙니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse 설정 — 키가 없으면 비활성 상태로 둡니다.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from langchain.tools import tool
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

## 12.1 도구는 “함수”가 아니라 “계약”입니다

LLM은 Python 구현이 아니라 이름, 설명, 인자 schema를 보고 도구를 선택합니다.

In [ ]:
@tool
def summarize_note(text: str, max_bullets: int = 3) -> str:
    """Summarize a note into a short Korean bullet list."""
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    return "\n".join(f"- {s}" for s in sentences[:max_bullets])

print(summarize_note.invoke({"text": "A. B. C.", "max_bullets": 2}))

## 12.2 도구 schema 점검

교육 자료에서는 tool schema를 출력해 학습자가 “모델이 무엇을 보는지” 이해하게 합니다.

In [ ]:
schema = summarize_note.args_schema.model_json_schema()

print("tool name:", summarize_note.name)
print("description:", summarize_note.description)
print("properties:", sorted(schema["properties"]))

## 12.3 도구 위험도 분류

도구를 만들 때는 실행 전 approval이 필요한지 먼저 분류합니다.

In [ ]:
tool_policy = {
    "summarize_note": "allow",
    "write_file": "approve",
    "execute_shell": "deny-or-sandbox",
}

for name, policy in tool_policy.items():
    print(f"{name:16s} -> {policy}")

## 12.4 모델 설정은 호출부에서 통일합니다

이 저장소는 교육 기본 모델을 `gpt-5.4`로 둡니다. 실제 실행 전에는 provider key와 비용 정책을 확인합니다.

In [ ]:
MODEL_NAME = os.getenv("COURSE_MODEL", "gpt-5.4")
agent_config = {
    "model": MODEL_NAME,
    "tools": [summarize_note],
    "backend": FilesystemBackend(root_dir=".", virtual_mode=True),
}
agent_config["model"]

## 12.5 에이전트 생성은 실행과 분리합니다

아래 셀은 호출하지 않고 agent 객체만 구성합니다. 실제 `.invoke()`는 API key, 비용, 권한 정책을 확인한 뒤 실행합니다.

In [ ]:
agent = create_deep_agent(
    model=agent_config["model"],
    tools=agent_config["tools"],
    backend=agent_config["backend"],
    system_prompt="You are a concise course assistant.",
)

type(agent).__name__

## 12.6 설계 점검표

| 질문 | 기준 |
|---|---|
| 모델 | provider, 비용, latency, context window가 맞는가? |
| 도구 | 이름과 docstring이 구체적인가? |
| 권한 | read/write/execute 위험도가 분리됐는가? |
| backend | local, store, sandbox 중 목적에 맞는가? |

---

## 정리

| 항목 | 내용 |
|---|---|
| **다룬 기술** | `@tool`, tool schema, `create_deep_agent`, `FilesystemBackend` |
| **핵심 개념** | Deep Agent의 품질은 model보다 tool 계약과 권한 경계에서 크게 갈립니다. |
| **다음 단계** | `13_programmatic_subagents.ipynb`, `14_event_streaming.ipynb` |

**참고 문서:**
- `docs/deepagents/models.md`
- `docs/deepagents/tools.md`
- `docs/deepagents/permissions.md`
- `docs/deepagents/backends.md`